# Session 2 · Part 3 — Train DGAT end to end

**Goal:** connect the four modules to five losses, four optimizers, backpropagation, evaluation, and
checkpoints. The complete official run is opt-in because conference laptops should use the lightweight
environment and verified precomputed predictions. The training code remains visible and runnable in the
separate `eccb-dgat-official` environment.


In [ ]:
from pathlib import Path
import sys

current = Path.cwd().resolve()
for candidate in (current, *current.parents):
    if (candidate / "src" / "dgat_tutorial").is_dir():
        tutorial_root = candidate
        break
else:
    raise FileNotFoundError("Start Jupyter inside the hands-on_tutorial directory.")

sys.path.insert(0, str(tutorial_root / "src"))

from dgat_tutorial.checkpoints import tutorial_paths, write_checkpoint

paths = tutorial_paths(tutorial_root)
print(f"Tutorial root: {paths.root}")


## 1. Decompose the training objective


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch

from dgat_tutorial.teaching import official_dgat_loss_table, weighted_training_objective

loss_table = official_dgat_loss_table()
loss_table


The scalar objective is

$L = \alpha L_{RNA-recon} + \beta L_{protein-recon} + \gamma L_{align}
+ \delta L_{RNA\rightarrow protein} + \epsilon L_{protein\rightarrow RNA}$.

The RNA→protein term directly trains the path used at inference. Reconstruction and reverse-direction
losses regularize the shared representation; alignment encourages paired spots to occupy similar latent
positions. Always log the components separately—a falling total can hide a failing task.


In [ ]:
example_logged_terms = dict(
    rna_reconstruction=0.42, protein_reconstruction=0.31, latent_alignment=0.08,
    protein_prediction=0.37, rna_prediction=0.46,
)
example_total = weighted_training_objective(**example_logged_terms)
print(f"Example arithmetic check (not a fitted result): total loss = {example_total:.2f}")


## 2. Inspect one real training step


In [ ]:
def dgat_training_step(batch, modules, optimizers, rmse_loss, mse_loss, weights):
    # Readable mirror of the upstream DGAT optimization step.
    encoder_rna, decoder_rna, encoder_protein, decoder_protein = modules
    for optimizer in optimizers:
        optimizer.zero_grad()

    x_rna = batch["mRNA"].x
    e_rna = batch[("mRNA", "mRNA_knn", "mRNA")].edge_index
    x_protein = batch["protein"].x
    e_protein = batch[("protein", "protein_knn", "protein")].edge_index

    z_rna = encoder_rna(x_rna, e_rna)
    z_protein = encoder_protein(x_protein, e_protein)

    losses = {
        "rna_reconstruction": rmse_loss(decoder_rna(z_rna), x_rna),
        "protein_reconstruction": rmse_loss(decoder_protein(z_protein), x_protein),
        "latent_alignment": mse_loss(z_rna, z_protein),
        "protein_prediction": rmse_loss(decoder_protein(z_rna), x_protein),
        "rna_prediction": rmse_loss(decoder_rna(z_protein), x_rna),
    }
    total = sum(weight * loss for weight, loss in zip(weights, losses.values()))
    total.backward()
    for module in modules:
        torch.nn.utils.clip_grad_norm_(module.parameters(), max_norm=1.0)
    for optimizer in optimizers:
        optimizer.step()
    return {name: float(value.detach()) for name, value in losses.items()} | {"total": float(total.detach())}

print("The function is defined for inspection. It is called by the epoch loop only in the official environment.")


### Figure 7 — Training, validation, and checkpoint workflow


In [ ]:
fig, ax = plt.subplots(figsize=(12, 3.2)); ax.set_xlim(0, 12); ax.set_ylim(0, 3); ax.axis("off")
labels = ["paired samples", "normalize + graphs", "forward pass", "5 losses", "backprop + clip", "validate", "save best"]
colors = ["#d9eaf7", "#d9eaf7", "#e4d7f4", "#fde2cd", "#f5b97f", "#d7ecd9", "#eeeeee"]
xs = [0.15, 1.9, 3.65, 5.4, 7.15, 8.9, 10.65]
for x, label, color in zip(xs, labels, colors):
    patch = FancyBboxPatch((x, 1.0), 1.25, 0.85, boxstyle="round,pad=0.04", facecolor=color, edgecolor="#333")
    ax.add_patch(patch); ax.text(x+0.625, 1.425, label, ha="center", va="center", fontsize=8.5)
for left in xs[:-1]:
    ax.add_patch(FancyArrowPatch((left+1.25,1.425),(left+1.75,1.425),arrowstyle="->",mutation_scale=11))
ax.add_patch(FancyArrowPatch((9.5,0.95),(3.9,0.85),connectionstyle="arc3,rad=-0.22",arrowstyle="->",mutation_scale=12))
ax.text(6.7, 0.22, "repeat epochs; stop/select using held-out samples", ha="center", fontsize=9)
workflow_path = paths.figures / "session02_training_workflow.png"
fig.savefig(workflow_path, dpi=180, bbox_inches="tight"); plt.show()


## 3. Run the official training workflow (opt in)


In [ ]:
RUN_FULL_TRAINING = False  # set True only in the eccb-dgat-official environment

if RUN_FULL_TRAINING:
    import anndata as ad
    import importlib
    import sys

    dgat_repo = paths.root / "external" / "DGAT"
    training_data = paths.root / "external" / "DGAT_assets" / "DGAT_training_datasets"
    if not dgat_repo.is_dir() or not training_data.is_dir():
        raise FileNotFoundError("Clone official DGAT and download DGAT_training_datasets first.")
    sys.path.insert(0, str(dgat_repo))
    train_and_predict = importlib.import_module("Model.Train_and_Predict")

    # Replace the glob patterns only if your asset release uses different names.
    rna_files = sorted(training_data.rglob("*RNA*.h5ad"))
    protein_files = sorted(training_data.rglob("*protein*.h5ad")) + sorted(training_data.rglob("*ADT*.h5ad"))
    if not rna_files or not protein_files:
        raise FileNotFoundError("Could not find paired RNA and protein/ADT H5AD files.")
    train_rna = [ad.read_h5ad(path) for path in rna_files]
    train_protein = [ad.read_h5ad(path) for path in protein_files]
    if len(train_rna) != len(train_protein):
        raise ValueError("Pair RNA and protein files by sample before training.")

    model_components = train_and_predict.train(train_rna, train_protein, str(paths.processed_data / "dgat_pyg"))
    print(model_components.keys())
else:
    print("Full training skipped. Continue to Part 4 for verified pretrained predictions.")


## 4. What a trustworthy training run must save

Save train/validation sample IDs, common gene/protein lists, preprocessing parameters, random seed,
per-epoch component losses, validation correlations, and all four state dictionaries. Split by biological
sample—not random spots—to avoid spatial and donor leakage. The tutorial's committed predictions remain
separate from observed evaluation proteins.


In [ ]:
loss_path = paths.results / "session02_dgat_loss_terms.csv"
loss_table.to_csv(loss_path, index=False)
manifest = write_checkpoint(
    "2.3", [loss_path, workflow_path],
    summary={"loss_terms": len(loss_table), "full_training_executed": RUN_FULL_TRAINING}, start=paths.root,
)
print(f"Checkpoint written: {manifest}")


## Check

Before moving on, explain why validation must hold out whole samples, which loss directly supervises
protein imputation, and which two modules are needed for RNA-only inference.
